In [2]:
from modeling import Bi_llm_head_EncoderModel
from safetensors import safe_open
from transformers import AutoModel, AutoTokenizer

# 假设自定义模型 Bi_llm_head_EncoderModel 已经定义好
model_path = '/home/admin/workspace/aop_lab/query_rec/experiments/llm_embedding/ckpt_tmp/checkpoint-9'

# 1. 首先加载模型结构
model = Bi_llm_head_EncoderModel(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

safe_tensor_path = model_path + '/model.safetensors'
model_tensors = {}
with safe_open(safe_tensor_path, framework='pt',device='cpu') as f:
    for k in f.keys():
        model_tensors[k] = f.get_tensor(k)
model.load_state_dict(model_tensors)

print(tokenizer)
print(model)

/opt/conda/envs/python3.8/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Qwen2TokenizerFast(name_or_path='/home/admin/workspace/aop_lab/query_rec/experiments/llm_embedding/ckpt_tmp/checkpoint-9', vocab_size=151643, model_max_length=131072, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>', '<|emb_0|>']}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|emb_0|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
Bi_llm_head_EncoderModel(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (lay

In [21]:
def encode(features, llm_embedding_token_type, normlized):
        embeddings = []
        psg_out = model.model(**features, return_dict=True)
        psg_out = model.lm_head(psg_out.last_hidden_state) # bsz seq_len head_size
        # 提取embeddings 
        total_len = features['attention_mask'].sum(dim=-1)  # bsz

        if llm_embedding_token_type == 'eos':
            # emb_token idx 
            emb_token_idx = total_len - 1
        elif llm_embedding_token_type == 'special':
            emb_token_idx = total_len - 2 
        else:
            raise ValueError('only support [eos, special], contact with hpc for more tech supports')

        bsz = psg_out.size(0)
        batch_indices = torch.arange(bsz)

        # bsz, head_size
        p_reps = psg_out[batch_indices, emb_token_idx]
        print(f'model.normlized :{model.normlized}')
        if normlized:
            p_reps = torch.nn.functional.normalize(p_reps, dim=-1)
        return p_reps.contiguous()


def encode_sentences(model, sentences, tokenizer, max_length=64):
    prompt = '<|im_start|>将下面这个query压缩成一个单词\nquery：{query}\n压缩后的单词：<|emb_0|><|im_end|>'
    prompt_list = []
    for data in sentences:
        prompt_list.append(prompt.format(query=data))
    qp_collated = tokenizer(
        prompt_list,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    return encode(qp_collated, 'special', True)

In [14]:
print(model.llm_embedding_token_type)

eos


In [26]:
a = tokenizer.encode('<|im_start|>将下面这个query压缩成一个单词\nquery：白色连衣裙\n压缩后的单词：<|emb_0|><|im_end|>')
print(a)
print(tokenizer.decode([151646]))

[151644, 44063, 100431, 99487, 1631, 107872, 12857, 46944, 110011, 198, 1631, 5122, 102440, 54926, 99741, 102807, 198, 107872, 104813, 110011, 5122, 151646, 151645]
<|emb_0|>


In [20]:
import torch
import torch.nn.functional as F
sents = ['白色连衣裙', '连衣裙白色']
res = encode_sentences(model, sents, tokenizer, max_length=64)
print(res.shape)
print(res[0] @ res[1])
print(F.cosine_similarity(res[0], res[1], dim=0))

model.llm_embedding_token_type :eos
model.normlized :False
torch.Size([2, 128])
tensor(0.9952, grad_fn=<SumBackward1>)


In [5]:
tokenizer.decode(100968)
# tokenizer.encode('最终结论：合适')


'合适'

In [27]:
import torch
import torch.nn.functional as F
sents = ['白色连衣裙', '连衣裙白色']
res = encode_sentences(model, sents, tokenizer, max_length=64)
print(type(res))
print(res.shape)
print(res[0] @ res[1])
# print(F.cosine_similarity(res[0], res[1], dim=0))

model.normlized :False
<class 'torch.Tensor'>
torch.Size([2, 128])
tensor(0.9914, grad_fn=<DotBackward0>)
